# DATASET

### Imports

In [ ]:
import pandas as pd
import numpy as np
import chess
from scipy import sparse


### Dataset

In [3]:
raw_data = pd.read_csv('chessData.csv')
labels = raw_data["Evaluation"]
moves = raw_data["FEN"]

### Key Functions

In [ ]:

def centipawns_to_win_probability(centipawns : str) -> float:
    if '#' in centipawns:
        return 1.0 if '+' in centipawns else 0.0
    return 1 / (1 + np.exp(-int(centipawns) / 150))

def FEN_to_input_vector(fen : str) -> np.ndarray:
    board = chess.Board(fen)
    turn = board.turn 
    
    input_vector = np.zeros(768, dtype=np.float32) # Moins de RAM !
        
    # Optimisation : piece_map() ne boucle que sur les cases occupées (max 32) au lieu de 64
    for square, piece in board.piece_map().items():
        display_square = square if turn == chess.WHITE else chess.square_mirror(square)
        
        piece_type = piece.piece_type - 1
        # Si c'est la pièce du joueur dont c'est le tour, offset = 0, sinon 6
        color_offset = 0 if piece.color == turn else 6               
                    
        input_vector[display_square * 12 + color_offset + piece_type] = 1   
            
    return input_vector

### Labels

In [5]:
LABELS = labels.to_numpy()

labels_probabilities = np.vectorize(centipawns_to_win_probability)

LABELS = labels_probabilities(LABELS)



In [ ]:
# MOVES = moves.to_numpy()

# MOVES = [FEN_to_input_vector(fen) for fen in MOVES]

In [6]:
np.savez('labels.npz', Y=LABELS)

### Moves

In [ ]:

def fens_to_sparse_matrix(fens):
    rows = []
    cols = []
    data = []
    for i, fen in enumerate(fens):
        board = chess.Board(fen)
        turn = board.turn
        
        for square, piece in board.piece_map().items():
            # Miroir si c'est aux noirs de jouer pour la symétrie
            display_square = square if turn == chess.WHITE else chess.square_mirror(square)
            piece_type = piece.piece_type - 1
            color_offset = 0 if piece.color == turn else 6
            
            index = display_square * 12 + color_offset + piece_type
            
            rows.append(i)
            cols.append(index)
            data.append(1.0)
            
    return sparse.csr_matrix((data, (rows, cols)), shape=(len(fens), 768), dtype=np.float32)

MOVES_sparse = fens_to_sparse_matrix(moves.to_numpy())

sparse.save_npz('moves.npz', MOVES_sparse)

### Full Code

In [ ]:
import pandas as pd
import numpy as np
import chess
from scipy import sparse

raw_data = pd.read_csv('chessData.csv')
labels = raw_data["Evaluation"]
moves = raw_data["FEN"]

def centipawns_to_win_probability(centipawns : str) -> float:
    if '#' in centipawns:
        return 1.0 if '+' in centipawns else 0.0
    return 1 / (1 + np.exp(-int(centipawns) / 150))

def FEN_to_input_vector(fen : str) -> np.ndarray:
    board = chess.Board(fen)
    turn = board.turn 
    
    input_vector = np.zeros(768, dtype=np.float32) # Moins de RAM !
        
    # Optimisation : piece_map() ne boucle que sur les cases occupées (max 32) au lieu de 64
    for square, piece in board.piece_map().items():
        display_square = square if turn == chess.WHITE else chess.square_mirror(square)
        
        piece_type = piece.piece_type - 1
        # Si c'est la pièce du joueur dont c'est le tour, offset = 0, sinon 6
        color_offset = 0 if piece.color == turn else 6               
                    
        input_vector[display_square * 12 + color_offset + piece_type] = 1   
            
    return input_vector

LABELS = labels.to_numpy()

labels_probabilities = np.vectorize(centipawns_to_win_probability)

LABELS = labels_probabilities(LABELS)


def fens_to_sparse_matrix(fens):
    rows = []
    cols = []
    data = []
    for i, fen in enumerate(fens):
        board = chess.Board(fen)
        turn = board.turn
        
        for square, piece in board.piece_map().items():
            # Miroir si c'est aux noirs de jouer pour la symétrie
            display_square = square if turn == chess.WHITE else chess.square_mirror(square)
            piece_type = piece.piece_type - 1
            color_offset = 0 if piece.color == turn else 6
            
            index = display_square * 12 + color_offset + piece_type
            
            rows.append(i)
            cols.append(index)
            data.append(1.0)
            
    return sparse.csr_matrix((data, (rows, cols)), shape=(len(fens), 768), dtype=np.float32)

MOVES_sparse = fens_to_sparse_matrix(moves.to_numpy())

sparse.save_npz('moves.npz', MOVES_sparse)
np.savez('labels.npz', Y=LABELS)

### Généré 

In [ ]:
import pandas as pd
import numpy as np
from scipy import sparse
import time

# --- CONFIGURATION ---
FILE_PATH = 'chessData.csv'
CHUNK_SIZE = 500_000        # Plus gros chunks = moins d'overhead pandas
SCALING_FACTOR = 150

# --- LOOKUP TABLE pré-calculée (évite des dicts/ifs dans la boucle chaude) ---
# piece_char -> (piece_type 0-5, is_white)
_P = {}
for _i, _c in enumerate('PNBRQK'):
    _P[_c] = (_i, True)
    _P[_c.lower()] = (_i, False)

def process_chunk_fast(fens, evals):
    """
    Parse les FEN manuellement (sans chess.Board = ~10x plus rapide)
    et calcule les labels de manière vectorisée.
    """
    n = len(fens)
    rows, cols = [], []
    vals = np.empty(n, dtype=np.float64)
    white_turn = np.empty(n, dtype=np.bool_)

    for i in range(n):
        fen = fens[i]
        ev = str(evals[i])

        # --- Tour : 1 seul char après le premier espace ---
        sp = fen.index(' ')
        is_w = fen[sp + 1] == 'w'
        white_turn[i] = is_w

        # --- Eval rapide (try/except > regex pour le cas courant) ---
        if '#' in ev:
            vals[i] = 10000.0 if '+' in ev else -10000.0
        else:
            try:
                vals[i] = int(ev)
            except ValueError:
                cleaned = ''.join(c for c in ev if c.isdigit() or c == '-')
                vals[i] = int(cleaned) if cleaned and cleaned != '-' else 0

        # --- Parsing FEN manuel (contourne chess.Board) ---
        sq = 56  # a8
        for ch in fen[:sp]:
            if ch == '/':
                sq -= 16          # descend d'un rang
            elif '1' <= ch <= '8':
                sq += ord(ch) - 48  # saute N cases (plus rapide que int())
            else:
                pt, iw = _P[ch]
                dsq = sq if is_w else (sq ^ 56)          # miroir vertical si noirs
                co = 0 if (iw == is_w) else 6             # 0 = ma pièce, 6 = adverse
                rows.append(i)
                cols.append(dsq * 12 + co + pt)
                sq += 1

    # --- Labels vectorisés (sigmoid sur tout le chunk d'un coup) ---
    vals[~white_turn] *= -1                               # perspective subjective
    labels = (1.0 / (1.0 + np.exp(-vals / SCALING_FACTOR))).astype(np.float32)

    data = np.ones(len(rows), dtype=np.float32)           # features binaires
    mat = sparse.csr_matrix((data, (rows, cols)), shape=(n, 768), dtype=np.float32)
    return mat, labels

# --- BOUCLE PRINCIPALE ---
t0 = time.perf_counter()
all_sparse = []
all_labels = []
total = 0

print("Début du traitement optimisé...")
reader = pd.read_csv(FILE_PATH, chunksize=CHUNK_SIZE)

for chunk in reader:
    mat, labels = process_chunk_fast(chunk['FEN'].values, chunk['Evaluation'].values)
    all_sparse.append(mat)
    all_labels.append(labels)
    total += len(labels)
    elapsed = time.perf_counter() - t0
    print(f"  {total:>10,} positions  |  {elapsed:.1f}s  |  {total/elapsed:,.0f} pos/s")

# --- Assemblage final & sauvegarde ---
print("Assemblage final...")
final_moves = sparse.vstack(all_sparse, format='csr')
final_labels = np.concatenate(all_labels)

sparse.save_npz('moves.npz', final_moves)
np.savez('labels.npz', Y=final_labels)

dt = time.perf_counter() - t0
print(f"\nTerminé ! {final_moves.shape[0]:,} positions sauvegardées en {dt:.1f}s ({final_moves.shape[0]/dt:,.0f} pos/s)")


Début du traitement optimisé...
     500,000 positions  |  10.0s  |  49,919 pos/s
   1,000,000 positions  |  19.9s  |  50,325 pos/s
   1,500,000 positions  |  29.7s  |  50,476 pos/s
   2,000,000 positions  |  40.8s  |  49,028 pos/s
   2,500,000 positions  |  50.3s  |  49,657 pos/s
   3,000,000 positions  |  59.7s  |  50,260 pos/s
   3,500,000 positions  |  69.2s  |  50,549 pos/s
   4,000,000 positions  |  78.3s  |  51,087 pos/s
   4,500,000 positions  |  87.3s  |  51,529 pos/s
   5,000,000 positions  |  95.4s  |  52,426 pos/s
   5,500,000 positions  |  103.2s  |  53,301 pos/s
   6,000,000 positions  |  111.4s  |  53,875 pos/s
   6,500,000 positions  |  119.5s  |  54,402 pos/s
   7,000,000 positions  |  128.3s  |  54,563 pos/s
   7,500,000 positions  |  137.4s  |  54,603 pos/s
   8,000,000 positions  |  147.8s  |  54,114 pos/s
   8,500,000 positions  |  158.6s  |  53,604 pos/s
   9,000,000 positions  |  168.5s  |  53,404 pos/s
   9,500,000 positions  |  177.9s  |  53,405 pos/s
  10,000,

In [4]:
sparse.save_npz('moves.npz', final_moves)
np.savez('labels.npz', Y=final_labels)

In [ ]:
import pandas as pd
import numpy as np
from scipy import sparse
import time

# --- CONFIGURATION ---
FILE_PATH = 'chessData.csv'
CHUNK_SIZE = 500_000        # Plus gros chunks = moins d'overhead pandas
SCALING_FACTOR = 150

# --- LOOKUP TABLE pré-calculée (évite des dicts/ifs dans la boucle chaude) ---
# piece_char -> (piece_type 0-5, is_white)
_P = {}
for _i, _c in enumerate('PNBRQK'):
    _P[_c] = (_i, True)
    _P[_c.lower()] = (_i, False)

def process_chunk_fast(fens, evals):
    """
    Parse les FEN manuellement (sans chess.Board = ~10x plus rapide)
    et calcule les labels de manière vectorisée.
    """
    n = len(fens)
    rows, cols = [], []
    vals = np.empty(n, dtype=np.float64)
    white_turn = np.empty(n, dtype=np.bool_)

    for i in range(n):
        fen = fens[i]
        ev = str(evals[i])

        # --- Tour : 1 seul char après le premier espace ---
        sp = fen.index(' ')
        is_w = fen[sp + 1] == 'w'
        white_turn[i] = is_w

        # --- Eval rapide (try/except > regex pour le cas courant) ---
        if '#' in ev:
            vals[i] = 10000.0 if '+' in ev else -10000.0
        else:
            try:
                vals[i] = int(ev)
            except ValueError:
                cleaned = ''.join(c for c in ev if c.isdigit() or c == '-')
                vals[i] = int(cleaned) if cleaned and cleaned != '-' else 0

        # --- Parsing FEN manuel (contourne chess.Board) ---
        sq = 56  # a8
        for ch in fen[:sp]:
            if ch == '/':
                sq -= 16          # descend d'un rang
            elif '1' <= ch <= '8':
                sq += ord(ch) - 48  # saute N cases (plus rapide que int())
            else:
                pt, iw = _P[ch]
                dsq = sq if is_w else (sq ^ 56)          # miroir vertical si noirs
                co = 0 if (iw == is_w) else 6             # 0 = ma pièce, 6 = adverse
                rows.append(i)
                cols.append(dsq * 12 + co + pt)
                sq += 1

    # --- Labels vectorisés (sigmoid sur tout le chunk d'un coup) ---
    vals[~white_turn] *= -1                               # perspective subjective
    labels = (1.0 / (1.0 + np.exp(-vals / SCALING_FACTOR))).astype(np.float32)

    data = np.ones(len(rows), dtype=np.float32)           # features binaires
    mat = sparse.csr_matrix((data, (rows, cols)), shape=(n, 768), dtype=np.float32)
    return mat, labels

# --- BOUCLE PRINCIPALE ---
t0 = time.perf_counter()
all_sparse = []
all_labels = []
total = 0

print("Début du traitement optimisé...")
reader = pd.read_csv(FILE_PATH, chunksize=CHUNK_SIZE)

for chunk in reader:
    mat, labels = process_chunk_fast(chunk['FEN'].values, chunk['Evaluation'].values)
    all_sparse.append(mat)
    all_labels.append(labels)
    total += len(labels)
    elapsed = time.perf_counter() - t0
    print(f"  {total:>10,} positions  |  {elapsed:.1f}s  |  {total/elapsed:,.0f} pos/s")

# --- Assemblage final & sauvegarde ---
print("Assemblage final...")
final_moves = sparse.vstack(all_sparse, format='csr')
final_labels = np.concatenate(all_labels)

sparse.save_npz('moves_halfKP.npz', final_moves)
np.savez('labels_halfKP.npz', Y=final_labels)

dt = time.perf_counter() - t0
print(f"\nTerminé ! {final_moves.shape[0]:,} positions sauvegardées en {dt:.1f}s ({final_moves.shape[0]/dt:,.0f} pos/s)")
